In [ ]:
# Purpose - makes spacetime plots of the data

fis = [63, 23] #[23, 63] #[26, 63] #[23, 63] #63 #6 #25

In [ ]:
# Imports and Folder Selection

import sys

import numpy as np
import pandas as pd
import xarray as xr

import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from graphet import Data
from graphet.plugins import TristanV2 

sys.path.append('..')
from useful_funcs import *
sys.path.append('../fmgmt')
from TristanRun import TristanRun
#from FieldLineHelper import FieldLineHelper

plt.close('all')
%matplotlib widget
set_standard_plot_params()

# Read Input File Data

runs = [TristanRun(findex) for findex in fis]

print(runs[1].sigma)

In [ ]:
# Configure Data

midlines = [xr.open_dataset(f"{TristanRun.find_folderpath(findex)}/output/cut_flds/midline.hdf5", engine='h5netcdf') for findex in fis]

durations = [len(midlines.t) * run.beta_A for midlines, run in zip(midlines, runs)]
print(f"Before: {durations}")

min_dur = min(durations)

midlines = [midline.isel(t=slice(0, int(min_dur / run.beta_A))) for midline, run in zip(midlines, runs)]

durations = [len(midlines.t) * run.beta_A for midlines, run in zip(midlines, runs)]
print(f"After: {durations}")

In [ ]:
#midlines = [d.fields.isel(x = len(d.fields.x) // 2).isel(z = 0) for d in ds]
print(midlines[0])

In [ ]:
print(midlines[1])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5), sharey = True)
plt.subplots_adjust(wspace=0)

fig_labels = ["(a)", "(b)"]


for i in range(2):
    run = runs[i]
    midline = midlines[i]
    ax = [ax1, ax2][i]

    xa, ya = [0.02, 0.99]

    ax.text(xa, ya, f"$\sigma$ = {run.sigma}", color='black', ha='left', va='top',
            transform=ax.transAxes, fontsize = 20, weight = "bold",
            bbox=dict(boxstyle='square,pad=0.3', edgecolor='white', facecolor='white', alpha=0.8))

    t_max = run.beta_A * run.cc / run.c_omp * run.output_interval * len(midline.t)
    #t_max = len(midline.t)

    im = ax.imshow(np.flipud(midline.dens), aspect='auto', extent=[midline.y.min(), midline.y.max(), 0, t_max], 
                   vmin=0, vmax=10)
    #fig.colorbar(im, ax=ax1, label='$n / n_0$')
    ax.set_xlabel('$y$ [$c / \omega_p$]')
    if (i == 0):
        ax.set_ylabel('$v_A t$ [$c / \omega_p$]')
    #ax.set_title(f"{fis[i]} sigma = {run.sigma}")

    ax.text(0.02, 0.02, fig_labels[i], color='white', ha='left', va='bottom',
        transform=ax.transAxes, fontsize=18, weight="bold",
        bbox=dict(boxstyle='square,pad=0.3', edgecolor='none', facecolor='black', alpha=0.5))
    
cbar = fig.colorbar(im, ax=[ax1, ax2], location='right', fraction=0.046, pad=0.04)
cbar.set_label('$n / n_0$')

save_polished_fig("spacetime")

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(np.flipud(midlines[1].dens), aspect='auto', vmin=0, vmax=10)
plt.axis('off')
plt.xticks([])
plt.yticks([])
plt.show()

In [ ]:
midlines['edens'] = midlines.dens * (midlines.epsilon - 1)

plt.figure(figsize=(10, 6))
t_max = (d.fields.t.max() - d.fields.t.min()) * run.gyrofreq
plt.imshow(np.flipud(midlines.edens), aspect='auto', extent=[d.fields.y.min(), d.fields.y.max(), 0, t_max], vmin = 0, vmax = 20)
plt.colorbar(label='KE Density')
plt.xlabel('$x$ [$c / \omega_p$]')
plt.ylabel('$\Omega t$')
plt.title(boilerplate)
plt.show()